# Priprava datasetu `event_omni`

Notebook vytvori eventovy dataset z `allDST_omni.csv` podla intervalov v `zoznam_udalosti_final.csv` a pripravi target stlpce `DST+1` az `DST+6`.

## Vstupy a vystupy

- Vstup 1: `../1_priprava_a_rozdelenie_dat/allDST_omni.csv`
- Vstup 2: `../0_datasety/zoznam_udalosti_final.csv`
- Vystup 1: `../0_datasety/event_omni.csv`
- Vystup 2: `../0_datasety/event_omni_prepared.csv`

In [1]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()
ALL_DST_PATH = BASE_DIR / "allDST_omni.csv"
EVENT_LIST_PATH = BASE_DIR.parent / "0_datasety" / "zoznam_udalosti_final.csv"
EVENT_OMNI_PATH = BASE_DIR.parent / "0_datasety" / "event_omni.csv"
EVENT_OMNI_PREPARED_PATH = BASE_DIR.parent / "0_datasety" / "event_omni_prepared.csv"

ALL_DST_PATH, EVENT_LIST_PATH, EVENT_OMNI_PATH, EVENT_OMNI_PREPARED_PATH

(PosixPath('/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/1_priprava_a_rozdelenie_dat/allDST_omni.csv'),
 PosixPath('/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/0_datasety/zoznam_udalosti_final.csv'),
 PosixPath('/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/0_datasety/event_omni.csv'),
 PosixPath('/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/0_datasety/event_omni_prepared.csv'))

In [2]:
df_all = pd.read_csv(ALL_DST_PATH)
df_intervals = pd.read_csv(EVENT_LIST_PATH)

df_all["time1"] = pd.to_datetime(df_all["time1"], utc=True)
df_intervals["Udalost_Od"] = pd.to_datetime(df_intervals["Udalost_Od"], utc=True)
df_intervals["Udalost_Do"] = pd.to_datetime(df_intervals["Udalost_Do"], utc=True)

df_all = df_all.sort_values("time1").reset_index(drop=True)
df_intervals = df_intervals.sort_values("Udalost_Od").reset_index(drop=True)

display(df_all.head())
display(df_intervals.head())

,Unnamed: 0,time1,bz_gsm,v,DST,DST+1,DST+2,DST+3,DST+4,DST+5,DST+6
0,0,1963-01-01 00:30:00+00:00,-0.2,285.0,-6,-6,-5.0,-5.0,-3.0,-3.0,-6.0
1,1,1963-01-01 01:30:00+00:00,-0.2,285.0,-5,-5,-5.0,-3.0,-3.0,-6.0,-8.0
2,2,1963-01-01 02:30:00+00:00,-0.2,285.0,-5,-5,-3.0,-3.0,-6.0,-8.0,-9.0
3,3,1963-01-01 03:30:00+00:00,-0.2,285.0,-3,-3,-3.0,-6.0,-8.0,-9.0,-6.0
4,4,1963-01-01 04:30:00+00:00,-0.2,285.0,-3,-3,-6.0,-8.0,-9.0,-6.0,-2.0


,ID,Udalost_Od,Udalost_Do,Min_DST_v_udalosti
0,1,1963-01-28 21:30:00+00:00,1963-02-02 21:30:00+00:00,-75
1,2,1963-02-08 01:30:00+00:00,1963-02-12 09:30:00+00:00,-62
2,3,1963-06-05 01:30:00+00:00,1963-06-09 08:30:00+00:00,-78
3,4,1963-08-17 22:30:00+00:00,1963-08-22 11:30:00+00:00,-84
4,5,1963-09-12 07:30:00+00:00,1963-09-30 21:30:00+00:00,-236


In [3]:
event_parts = []

for row in df_intervals.itertuples(index=False):
    part = df_all.loc[
        (df_all["time1"] >= row.Udalost_Od) & (df_all["time1"] <= row.Udalost_Do),
        ["time1", "bz_gsm", "v", "DST"],
    ].copy()
    part["event_no"] = int(row.ID)
    event_parts.append(part)

df_event_omni = pd.concat(event_parts, ignore_index=True)
df_event_omni = df_event_omni.rename(columns={"DST": "dst"})
df_event_omni = df_event_omni.sort_values(["event_no", "time1"]).reset_index(drop=True)

df_event_omni.to_csv(EVENT_OMNI_PATH, index=False)

print(f"Saved raw event dataset to: {EVENT_OMNI_PATH}")
print(f"Rows: {len(df_event_omni)}")
print(f"Events: {df_event_omni['event_no'].nunique()}")
display(df_event_omni.head())

Saved raw event dataset to: /home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/0_datasety/event_omni.csv
Rows: 158430
Events: 953


,time1,bz_gsm,v,dst,event_no
0,1963-01-28 21:30:00+00:00,-0.2,285.0,7,1
1,1963-01-28 22:30:00+00:00,-0.2,285.0,8,1
2,1963-01-28 23:30:00+00:00,-0.2,285.0,6,1
3,1963-01-29 00:30:00+00:00,-0.2,285.0,8,1
4,1963-01-29 01:30:00+00:00,-0.2,285.0,7,1


In [4]:
df_event_prepared = df_event_omni.copy()

for horizon in range(1, 7):
    df_event_prepared[f"DST+{horizon}"] = (
        df_event_prepared.groupby("event_no")["dst"].shift(-horizon)
    )

df_event_prepared.to_csv(EVENT_OMNI_PREPARED_PATH, index=False)

usable_rows = {
    f"DST+{h}": int(df_event_prepared[f"DST+{h}"].notna().sum())
    for h in range(1, 7)
}

print(f"Saved prepared event dataset to: {EVENT_OMNI_PREPARED_PATH}")
print("Usable rows per horizon:")
for key, value in usable_rows.items():
    print(f"  {key}: {value}")

display(df_event_prepared.head())

Saved prepared event dataset to: /home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/DP_Drengubiakova_Terezia_1/0_datasety/event_omni_prepared.csv
Usable rows per horizon:
  DST+1: 157477
  DST+2: 156524
  DST+3: 155571
  DST+4: 154618
  DST+5: 153665
  DST+6: 152712


,time1,bz_gsm,v,dst,event_no,DST+1,DST+2,DST+3,DST+4,DST+5,DST+6
0,1963-01-28 21:30:00+00:00,-0.2,285.0,7,1,8.0,6.0,8.0,7.0,9.0,11.0
1,1963-01-28 22:30:00+00:00,-0.2,285.0,8,1,6.0,8.0,7.0,9.0,11.0,12.0
2,1963-01-28 23:30:00+00:00,-0.2,285.0,6,1,8.0,7.0,9.0,11.0,12.0,13.0
3,1963-01-29 00:30:00+00:00,-0.2,285.0,8,1,7.0,9.0,11.0,12.0,13.0,12.0
4,1963-01-29 01:30:00+00:00,-0.2,285.0,7,1,9.0,11.0,12.0,13.0,12.0,10.0


In [5]:
summary = pd.DataFrame(
    {
        "rows": [len(df_event_prepared)],
        "events": [df_event_prepared["event_no"].nunique()],
        "time_min": [df_event_prepared["time1"].min()],
        "time_max": [df_event_prepared["time1"].max()],
        "dst_min": [df_event_prepared["dst"].min()],
        "dst_max": [df_event_prepared["dst"].max()],
    }
)

display(summary)

,rows,events,time_min,time_max,dst_min,dst_max
0,158430,953,1963-01-28 21:30:00+00:00,2026-01-13 15:30:00+00:00,-589,81
